# ENGG2112 — TACO Litter Object Detection with YOLOv8n

This notebook contains the final machine-learning workflow for the project:

1. Environment and Google Drive setup
2. Dataset preparation (TACO → six selected classes → YOLO format)
3. Three YOLOv8n training configurations
4. Validation-set comparison
5. Final held-out test evaluation
6. Qualitative error analysis and prediction examples

**Final selected model:** YOLOv8n, 960×960 input resolution, 50 epochs.

> Training cells are disabled by default because the trained weights are already saved in Google Drive.  
> Set the relevant flags to `True` only when you intentionally want to rebuild data, retrain, or rerun evaluation.

## 1. Environment setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip install -q ultralytics kagglehub opencv-python

In [ ]:
import os
import json
import shutil
from collections import Counter, defaultdict

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from IPython.display import Image as DisplayImage, display

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Project configuration

In [ ]:
DRIVE_PROJECT = "/content/drive/MyDrive/ENGG2112_Final_Project"

YOLO_ROOT = os.path.join(DRIVE_PROJECT, "taco_yolo")
YAML_PATH = os.path.join(YOLO_ROOT, "data.yaml")

BASELINE_PATH = os.path.join(
    DRIVE_PROJECT, "baseline", "weights", "best.pt"
)
EXP2_PATH = os.path.join(
    DRIVE_PROJECT, "runs", "img960", "weights", "best.pt"
)
FINAL_MODEL_PATH = os.path.join(
    DRIVE_PROJECT, "runs", "img960_ep50", "weights", "best.pt"
)

ANALYSIS_ROOT = os.path.join(DRIVE_PROJECT, "analysis")

CLASS_NAMES = [
    "Cigarette",
    "Plastic film",
    "Clear plastic bottle",
    "Drink can",
    "Plastic bottle cap",
    "Glass bottle",
]

os.makedirs(DRIVE_PROJECT, exist_ok=True)

print("Dataset YAML:", os.path.exists(YAML_PATH))
print("Baseline model:", os.path.exists(BASELINE_PATH))
print("Experiment 2 model:", os.path.exists(EXP2_PATH))
print("Final model:", os.path.exists(FINAL_MODEL_PATH))

## 3. Dataset preparation

The project uses the public **TACO (Trash Annotations in Context)** dataset.  
The original COCO annotations were filtered to six relatively common, clearly defined litter classes:

- Cigarette
- Plastic film
- Clear plastic bottle
- Drink can
- Plastic bottle cap
- Glass bottle

Ambiguous categories such as *Unlabeled litter* and *Other plastic* were excluded.

The prepared dataset uses a deterministic **70/15/15 train/validation/test split** with `random_state=2112`.

> The prepared dataset is already stored in Google Drive.  
> Leave `REBUILD_DATASET = False` unless the dataset needs to be regenerated from the original TACO files.

In [ ]:
REBUILD_DATASET = False

In [ ]:
if REBUILD_DATASET:
    import kagglehub

    # Download public TACO dataset
    taco_path = kagglehub.dataset_download("kneroma/tacotrashdataset")
    print("Downloaded TACO to:", taco_path)

    # Locate annotations.json
    annotation_path = None
    for root, _, files in os.walk(taco_path):
        if "annotations.json" in files:
            annotation_path = os.path.join(root, "annotations.json")
            break

    if annotation_path is None:
        raise FileNotFoundError("annotations.json was not found in the downloaded TACO dataset.")

    with open(annotation_path, "r", encoding="utf-8") as f:
        coco_data = json.load(f)

    # Image root is the folder containing annotations.json
    data_root = os.path.dirname(annotation_path)

    category_map = {
        category["id"]: category["name"]
        for category in coco_data["categories"]
    }

    category_counts = Counter(
        ann["category_id"] for ann in coco_data["annotations"]
    )
    category_stats = sorted(
        [
            (category_map[category_id], category_id, count)
            for category_id, count in category_counts.items()
        ],
        key=lambda x: x[2],
        reverse=True,
    )

    selected_names = CLASS_NAMES
    selected_categories = [
        (name, category_id, count)
        for name, category_id, count in category_stats
        if name in selected_names
    ]
    selected_categories = sorted(
        selected_categories,
        key=lambda x: selected_names.index(x[0]),
    )

    selected_category_ids = [
        category_id for _, category_id, _ in selected_categories
    ]
    old_to_new_id = {
        old_id: new_id
        for new_id, old_id in enumerate(selected_category_ids)
    }

    print("Selected classes:")
    for new_id, (name, old_id, count) in enumerate(selected_categories):
        print(f"{new_id}: {name:25s} TACO_ID={old_id:<3} count={count}")

    # Index annotations by image
    annotations_by_image = defaultdict(list)
    for ann in coco_data["annotations"]:
        annotations_by_image[ann["image_id"]].append(ann)

    # Keep images containing at least one selected class
    selected_image_ids = {
        ann["image_id"]
        for ann in coco_data["annotations"]
        if ann["category_id"] in selected_category_ids
    }
    selected_images = [
        img for img in coco_data["images"]
        if img["id"] in selected_image_ids
    ]

    # Deterministic 70/15/15 split
    train_images, temp_images = train_test_split(
        selected_images,
        test_size=0.30,
        random_state=2112,
    )
    val_images, test_images = train_test_split(
        temp_images,
        test_size=0.50,
        random_state=2112,
    )

    # Rebuild output folders from scratch
    if os.path.exists(YOLO_ROOT):
        shutil.rmtree(YOLO_ROOT)

    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(YOLO_ROOT, "images", split), exist_ok=True)
        os.makedirs(os.path.join(YOLO_ROOT, "labels", split), exist_ok=True)

    def coco_bbox_to_yolo(bbox, img_width, img_height):
        x, y, w, h = bbox
        x_center = (x + w / 2) / img_width
        y_center = (y + h / 2) / img_height
        return (
            x_center,
            y_center,
            w / img_width,
            h / img_height,
        )

    def prepare_split(images, split_name):
        converted_images = 0
        converted_boxes = 0

        for img_info in images:
            image_id = img_info["id"]
            file_name = img_info["file_name"]
            source_image = os.path.join(data_root, file_name)

            if not os.path.exists(source_image):
                continue

            labels = []
            for ann in annotations_by_image[image_id]:
                old_category_id = ann["category_id"]
                if old_category_id not in selected_category_ids:
                    continue

                new_category_id = old_to_new_id[old_category_id]
                xc, yc, w, h = coco_bbox_to_yolo(
                    ann["bbox"],
                    img_info["width"],
                    img_info["height"],
                )

                if w <= 0 or h <= 0:
                    continue

                labels.append(
                    f"{new_category_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}"
                )

            if not labels:
                continue

            new_image_name = file_name.replace("/", "_").replace("\\", "_")
            destination_image = os.path.join(
                YOLO_ROOT, "images", split_name, new_image_name
            )
            shutil.copy2(source_image, destination_image)

            label_name = os.path.splitext(new_image_name)[0] + ".txt"
            label_path = os.path.join(
                YOLO_ROOT, "labels", split_name, label_name
            )
            with open(label_path, "w", encoding="utf-8") as f:
                f.write("\n".join(labels))

            converted_images += 1
            converted_boxes += len(labels)

        print(
            f"{split_name}: {converted_images} images, "
            f"{converted_boxes} boxes"
        )

    prepare_split(train_images, "train")
    prepare_split(val_images, "val")
    prepare_split(test_images, "test")

    # Create YOLO data.yaml
    with open(YAML_PATH, "w", encoding="utf-8") as f:
        f.write(
            f"path: {YOLO_ROOT}\n"
            "train: images/train\n"
            "val: images/val\n"
            "test: images/test\n\n"
            "names:\n"
        )
        for i, name in enumerate(CLASS_NAMES):
            f.write(f"  {i}: {name}\n")

    print("\nCreated:", YAML_PATH)
else:
    print("Using the prepared YOLO dataset already stored in Google Drive.")

In [ ]:
# Verify the prepared dataset
if not os.path.exists(YAML_PATH):
    raise FileNotFoundError(
        f"{YAML_PATH} does not exist. "
        "Set REBUILD_DATASET=True and run the dataset-preparation cells."
    )

print(open(YAML_PATH, "r", encoding="utf-8").read())

for split in ["train", "val", "test"]:
    image_dir = os.path.join(YOLO_ROOT, "images", split)
    label_dir = os.path.join(YOLO_ROOT, "labels", split)
    print(
        f"{split:5s}: "
        f"{len(os.listdir(image_dir))} images, "
        f"{len(os.listdir(label_dir))} label files"
    )

### Final dataset split

- **Training:** 596 images, 1,341 annotated boxes
- **Validation:** 128 images, 314 annotated boxes
- **Test:** 128 images, 290 annotated boxes

## 4. Model experiments

All experiments use pretrained **YOLOv8n** weights.

The three recorded configurations were:

1. **Baseline:** 640×640, 20 epochs, batch size 16
2. **Experiment 2:** 960×960, 20 epochs, batch size 8
3. **Experiment 3:** 960×960, 50 epochs, batch size 8, explicit seed 2112

Because batch size and seed settings were not identical across all runs, performance differences are interpreted as **changes across configurations**, not as effects caused solely by resolution or training duration.

> The training cells below are retained for reproducibility and are disabled by default.

In [ ]:
RUN_TRAINING = False

### Experiment 1 — Baseline (640×640, 20 epochs)

In [ ]:
if RUN_TRAINING:
    baseline_train_model = YOLO("yolov8n.pt")
    baseline_train_model.train(
        data=YAML_PATH,
        epochs=20,
        imgsz=640,
        batch=16,
        device=0,
        seed=0,
        project=DRIVE_PROJECT,
        name="baseline",
    )
else:
    print("Skipped baseline training; using saved weights.")

### Experiment 2 — Higher resolution (960×960, 20 epochs)

In [ ]:
if RUN_TRAINING:
    exp2_train_model = YOLO("yolov8n.pt")
    exp2_train_model.train(
        data=YAML_PATH,
        epochs=20,
        imgsz=960,
        batch=8,
        device=0,
        seed=0,
        project=os.path.join(DRIVE_PROJECT, "runs"),
        name="img960",
    )
else:
    print("Skipped Experiment 2 training; using saved weights.")

### Experiment 3 — Longer training (960×960, 50 epochs)

In [ ]:
if RUN_TRAINING:
    exp3_train_model = YOLO("yolov8n.pt")
    exp3_train_model.train(
        data=YAML_PATH,
        epochs=50,
        imgsz=960,
        batch=8,
        device=0,
        seed=2112,
        project=os.path.join(DRIVE_PROJECT, "runs"),
        name="img960_ep50",
    )
else:
    print("Skipped Experiment 3 training; using saved weights.")

## 5. Validation experiment comparison

| Experiment | Resolution | Epochs | Precision | Recall | mAP50 | mAP50–95 |
|---|---:|---:|---:|---:|---:|---:|
| Baseline | 640 | 20 | 0.537 | 0.280 | 0.307 | 0.218 |
| Experiment 2 | 960 | 20 | 0.516 | 0.297 | 0.327 | 0.243 |
| Experiment 3 | 960 | 50 | 0.496 | 0.348 | 0.357 | 0.256 |

**Table 1. Validation performance across the three YOLOv8n training configurations.**

Experiment 3 achieved the highest validation recall, mAP@0.5 and mAP@0.5:0.95, and was selected for the final held-out test evaluation. Because the configurations also differed in batch size and seed settings, these differences should not be attributed solely to resolution or training duration.

In [ ]:
# Load the three saved models
for path, label in [
    (BASELINE_PATH, "Baseline"),
    (EXP2_PATH, "Experiment 2"),
    (FINAL_MODEL_PATH, "Experiment 3 / final"),
]:
    print(f"{label:24s}: {os.path.exists(path)}")

baseline_model = YOLO(BASELINE_PATH)
exp2_model = YOLO(EXP2_PATH)
final_model = YOLO(FINAL_MODEL_PATH)

## 6. Validation-set qualitative error analysis

The comparison below uses one fixed validation image so that the same example is reproduced each time. It compares:

- Ground-truth annotations
- Baseline 640×640 prediction
- Experiment 2 960×960 prediction

This figure is a **validation-set qualitative comparison**, not a final test result.

In [ ]:
sample_file = "batch_8_000027.jpg"
sample_path = os.path.join(YOLO_ROOT, "images", "val", sample_file)
label_path = os.path.join(
    YOLO_ROOT,
    "labels",
    "val",
    os.path.splitext(sample_file)[0] + ".txt",
)

if not os.path.exists(sample_path):
    raise FileNotFoundError(sample_path)
if not os.path.exists(label_path):
    raise FileNotFoundError(label_path)

pred_640 = baseline_model.predict(
    source=sample_path,
    conf=0.25,
    verbose=False,
)
pred_960 = exp2_model.predict(
    source=sample_path,
    conf=0.25,
    verbose=False,
)

# Draw ground truth
image_bgr = cv2.imread(sample_path)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
h, w = image_rgb.shape[:2]
gt_image = image_rgb.copy()

with open(label_path, "r", encoding="utf-8") as f:
    for line in f:
        class_id, xc, yc, bw, bh = map(float, line.split())
        class_id = int(class_id)

        xc *= w
        yc *= h
        bw *= w
        bh *= h

        x1 = int(xc - bw / 2)
        y1 = int(yc - bh / 2)
        x2 = int(xc + bw / 2)
        y2 = int(yc + bh / 2)

        cv2.rectangle(gt_image, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(
            gt_image,
            CLASS_NAMES[class_id],
            (x1, max(y1 - 5, 12)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (255, 0, 0),
            1,
        )

img_640 = pred_640[0].plot()
img_960 = pred_960[0].plot()

plt.figure(figsize=(18, 6))

plt.subplot(1, 3, 1)
plt.imshow(gt_image)
plt.axis("off")
plt.title("Ground Truth")

plt.subplot(1, 3, 2)
plt.imshow(img_640[:, :, ::-1])
plt.axis("off")
plt.title("Baseline: 640 × 640")

plt.subplot(1, 3, 3)
plt.imshow(img_960[:, :, ::-1])
plt.axis("off")
plt.title("Experiment 2: 960 × 960")

plt.tight_layout()
plt.show()

In [ ]:
def print_predictions(result, model_name, names):
    print(f"=== {model_name} ===")
    if len(result.boxes) == 0:
        print("No objects detected.")
        return
    for box in result.boxes:
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])
        print(names[class_id], f"{confidence:.2%}")

print_predictions(pred_640[0], "Baseline 640", baseline_model.names)
print()
print_predictions(pred_960[0], "Experiment 2 — 960", exp2_model.names)

## 7. Final held-out test evaluation

The final model was selected **before** evaluating the held-out test set.

**Selected model:** YOLOv8n, 960×960 input resolution, 50 epochs.

In [ ]:
RUN_FINAL_TEST = False

if RUN_FINAL_TEST:
    test_metrics = final_model.val(
        data=YAML_PATH,
        split="test",
        plots=True,
        project=ANALYSIS_ROOT,
        name="final_test",
    )

    print("FINAL TEST RESULTS")
    print("Precision:", test_metrics.box.mp)
    print("Recall:", test_metrics.box.mr)
    print("mAP@0.5:", test_metrics.box.map50)
    print("mAP@0.5:0.95:", test_metrics.box.map)
else:
    print("Skipped rerunning final test; using the recorded held-out test results below.")

### Final held-out test results

| Metric | Test result |
|---|---:|
| Precision | 0.522 |
| Recall | 0.411 |
| mAP@0.5 | 0.409 |
| mAP@0.5:0.95 | 0.303 |

These values come from the **held-out test set**, not the validation set.

In [ ]:
# Display the saved final-test normalized confusion matrix if available
final_test_dir = os.path.join(ANALYSIS_ROOT, "final_test")
cm_path = os.path.join(final_test_dir, "confusion_matrix_normalized.png")

if os.path.exists(cm_path):
    display(DisplayImage(filename=cm_path))
else:
    print(
        "Final-test confusion matrix not found. "
        "Set RUN_FINAL_TEST=True once to regenerate it."
    )

## 8. Final test prediction examples

In [ ]:
prediction_dir = os.path.join(
    ANALYSIS_ROOT,
    "final_test_predictions",
)

if os.path.exists(prediction_dir):
    prediction_files = sorted(
        f for f in os.listdir(prediction_dir)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    )
    print("Saved prediction images:", len(prediction_files))

    for file in prediction_files[:5]:
        display(
            DisplayImage(
                filename=os.path.join(prediction_dir, file)
            )
        )
else:
    print(
        "No saved final-test predictions found. "
        "Run the optional prediction cell below to generate them."
    )

In [ ]:
GENERATE_TEST_PREDICTIONS = False

if GENERATE_TEST_PREDICTIONS:
    test_image_dir = os.path.join(YOLO_ROOT, "images", "test")
    final_model.predict(
        source=test_image_dir,
        conf=0.25,
        save=True,
        project=ANALYSIS_ROOT,
        name="final_test_predictions",
    )
    print("Final-test prediction images saved.")

## 9. Final model artifacts

The selected training run is stored under:

`ENGG2112_Final_Project/runs/img960_ep50/`

Important files:

- `weights/best.pt` — selected trained model
- `results.csv` — epoch-by-epoch training metrics
- `args.yaml` — recorded training configuration
- `analysis/final_test/` — held-out test evaluation plots
- `analysis/final_test_predictions/` — saved test-set prediction images

In [ ]:
artifact_paths = {
    "best.pt": FINAL_MODEL_PATH,
    "results.csv": os.path.join(
        DRIVE_PROJECT, "runs", "img960_ep50", "results.csv"
    ),
    "args.yaml": os.path.join(
        DRIVE_PROJECT, "runs", "img960_ep50", "args.yaml"
    ),
    "final_test": os.path.join(ANALYSIS_ROOT, "final_test"),
    "final_test_predictions": os.path.join(
        ANALYSIS_ROOT, "final_test_predictions"
    ),
}

for name, path in artifact_paths.items():
    print(f"{name:24s}: {os.path.exists(path)}  {path}")

## 10. Technical summary

The final system is a **general litter object-detection prototype** trained on six TACO classes.  
The selected YOLOv8n model uses a 960×960 input resolution and 50 training epochs.

The held-out test results were:

- Precision: **0.522**
- Recall: **0.411**
- mAP@0.5: **0.409**
- mAP@0.5:0.95: **0.303**

The model remains limited by missed small objects and false positives. Campus litter monitoring is therefore treated as a **potential application**, with human review appropriate for any practical decision-support workflow.